# Ingest data from Bronze to Silver

- Ref: [Merge operation in Delta table](https://delta.io/blog/2023-02-14-delta-lake-merge/)

In [1]:
import os
import sys
sys.path.append('/home/nblong/Dropbox/Project/ops/ml-ops/src/')

from dotenv import load_dotenv

load_dotenv('../../../../.env_file/infra.env'), load_dotenv('../../../../.env_file/app.env')
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

from silver.silver_map import *


In [ ]:
spark = (
    SparkSession.builder
    .appName('ingest data to silver zone')
    # delta lake support
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    # read/write S3
    .config("spark.hadoop.fs.s3a.access.key", os.getenv('AWS_ACCESS_KEY_ID'))
    .config("spark.hadoop.fs.s3a.secret.key", os.getenv('AWS_SECRET_ACCESS_KEY'))
    # read postgres
    .config(
        'spark.jars.packages',
        'org.apache.hadoop:hadoop-aws:3.4.2,'
        'software.amazon.awssdk:bundle:2.29.52,'
        'io.delta:delta-spark_4.1_2.13:4.3.0,'
    )
    .config("spark.databricks.delta.properties.defaults.enableChangeDataFeed", "true")
    # Hive metastore
    .config("spark.hadoop.hive.metastore.uris", "thrift://localhost:9084")
    .config("spark.sql.catalogImplementation", "hive")
    .enableHiveSupport()
    .getOrCreate()
)
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/16 08:39:57 WARN Utils: Your hostname, debian-nblong, resolves to a loopback address: 127.0.1.1; using 192.168.36.108 instead (on interface wlp0s20f3)
26/10/16 08:39:57 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/nblong/Dropbox/Project/ops/ml-ops/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/nblong/.ivy2.5.2/cache
The jars for the packages stored in: /home/nblong/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
software.amazon.awssdk#bundle added as a dependency
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-5c504c4f-5902-4ddb-8d67-047379feb4a3;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.4.2 in central
	found software.amazon.awssdk#bundl

In [ ]:
# trước đó tôi create table này nhưng không enable option để có thể CDC nó
# giờ tôi phải alter table
# phải sửa code create thành alter table
spark.sql("ALTER TABLE silver.nyc_taxi_stream SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
26/10/16 08:41:08 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


DataFrame[]

## Transfer

- Read from Bronze

In [3]:
df = spark.read.table('bronze.nyc_taxi')
df = df.filter((df['year']==2021) & (df['month']==10))

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


In [4]:
df.dtypes

[('vendor_id', 'string'),
 ('pickup_datetime', 'timestamp'),
 ('dropoff_datetime', 'timestamp'),
 ('passenger_count', 'double'),
 ('trip_distance', 'double'),
 ('rate_code_id', 'double'),
 ('store_and_fwd_flag', 'string'),
 ('pickup_longitude', 'double'),
 ('pickup_latitude', 'double'),
 ('dropoff_longitude', 'double'),
 ('dropoff_latitude', 'double'),
 ('pu_location_id', 'string'),
 ('do_location_id', 'string'),
 ('payment_type', 'string'),
 ('fare_amount', 'double'),
 ('extra', 'double'),
 ('mta_tax', 'double'),
 ('surcharge', 'double'),
 ('improvement_surcharge', 'double'),
 ('tip_amount', 'double'),
 ('tolls_amount', 'double'),
 ('total_amount', 'double'),
 ('congestion_surcharge', 'double'),
 ('airport_fee', 'double'),
 ('cbd_congestion_fee', 'double'),
 ('source_file', 'string'),
 ('source_created_datetime', 'timestamp'),
 ('record_key', 'string'),
 ('bronze_created_datetime', 'timestamp'),
 ('year', 'int'),
 ('month', 'int')]

In [7]:
df.filter(
    df['pickup_datetime'] > F.to_timestamp(F.lit('2009-01-01 00:00:00'))
).show()

26/06/22 06:30:00 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---------+-------------------+-------------------+---------------+------------------+------------+------------------+------------------+---------------+------------------+----------------+--------------+--------------+------------+-----------+-----+-------+---------+---------------------+----------+------------+------------+--------------------+-----------+------------------+----+-----+
|vendor_id|    pickup_datetime|   dropoff_datetime|passenger_count|     trip_distance|rate_code_id|store_and_fwd_flag|  pickup_longitude|pickup_latitude| dropoff_longitude|dropoff_latitude|pu_location_id|do_location_id|payment_type|fare_amount|extra|mta_tax|surcharge|improvement_surcharge|tip_amount|tolls_amount|total_amount|congestion_surcharge|airport_fee|cbd_congestion_fee|year|month|
+---------+-------------------+-------------------+---------------+------------------+------------+------------------+------------------+---------------+------------------+----------------+--------------+--------------

In [7]:
df.createOrReplaceTempView('nyc_taxi')
spark.sql("select count(*) from nyc_taxi where pickup_datetime > timestamp '2009-01-01 00:00:00' and pickup_datetime < timestamp '2009-01-02 00:00:00';").show()

26/06/22 03:44:37 ERROR Executor: Exception in task 0.0 in stage 3.0 (TID 2)
java.sql.SQLException: Query failed (#20260622_034437_00026_7qusb): line 4:63: Cannot apply operator: timestamp(3) < varchar(21)
	at io.trino.jdbc.ResultUtils.resultsException(ResultUtils.java:33)
	at io.trino.jdbc.TrinoResultSet.getColumns(TrinoResultSet.java:143)
	at io.trino.jdbc.TrinoResultSet.create(TrinoResultSet.java:49)
	at io.trino.jdbc.TrinoStatement.internalExecute(TrinoStatement.java:276)
	at io.trino.jdbc.TrinoStatement.execute(TrinoStatement.java:254)
	at io.trino.jdbc.TrinoPreparedStatement.executeQuery(TrinoPreparedStatement.java:149)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCRDD.$anonfun$compute$9(JDBCRDD.scala:336)
	at org.apache.spark.sql.execution.metric.SQLMetrics$.withTimingNs(SQLMetrics.scala:234)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCRDD.compute(JDBCRDD.scala:335)
	at org.apache.spark.rdd.RDD.computeOrReadCheckpoint(RDD.scala:374)
	at org.apache.spark.rdd

Py4JJavaError: An error occurred while calling o73.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 3.0 failed 1 times, most recent failure: Lost task 0.0 in stage 3.0 (TID 2) (ip-10-0-0-6.ap-northeast-1.compute.internal executor driver): java.sql.SQLException: Query failed (#20260622_034437_00026_7qusb): line 4:63: Cannot apply operator: timestamp(3) < varchar(21)
	at io.trino.jdbc.ResultUtils.resultsException(ResultUtils.java:33)
	at io.trino.jdbc.TrinoResultSet.getColumns(TrinoResultSet.java:143)
	at io.trino.jdbc.TrinoResultSet.create(TrinoResultSet.java:49)
	at io.trino.jdbc.TrinoStatement.internalExecute(TrinoStatement.java:276)
	at io.trino.jdbc.TrinoStatement.execute(TrinoStatement.java:254)
	at io.trino.jdbc.TrinoPreparedStatement.executeQuery(TrinoPreparedStatement.java:149)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCRDD.$anonfun$compute$9(JDBCRDD.scala:336)
	at org.apache.spark.sql.execution.metric.SQLMetrics$.with

- Cast some columns to int

In [10]:
cast_map = {
    'passenger_count': F.col('passenger_count').cast(T.IntegerType()),
    'pu_location_id' : F.col('pu_location_id').cast(T.IntegerType()),
    'do_location_id' : F.col('do_location_id').cast(T.IntegerType())
}
df = df.withColumns(cast_map)

In [4]:
def value_counts(df:DataFrame, col:str) -> DataFrame:
    return df.groupBy(col).count().orderBy('count')

In [5]:
def _transform(df:DataFrame)->DataFrame:
    # cast some cols to their natural datatype
    cast_map = {
        'passenger_count': F.col('passenger_count').cast(T.IntegerType()),
        'pu_location_id' : F.col('pu_location_id').cast(T.IntegerType()),
        'do_location_id' : F.col('do_location_id').cast(T.IntegerType()),
    }
    df = df.withColumns(cast_map)

    # payment_type
    df = (
        df
        .withColumn('payment_type', F.lower(F.trim(F.col('payment_type'))))
        .replace(payment_map, subset='payment_type')
    )
    # payment_type_id
    df = (
        df.withColumn('payment_type_id', F.col('payment_type'))
        .replace(inverse_payment_map, subset='payment_type_id')
    )

    # drop outlier for `passenger_count`
    df = df.filter(
        (F.col('passenger_count')<7) &
        (F.col('passenger_count')!=0)
    )

    # drop outlier for `rate_code_id` and cast to int
    df = (
        df.filter(
            ((F.col('rate_code_id')>=1) & (F.col('rate_code_id')<=6)) | (F.col('rate_code_id')==99) |
            (F.col('rate_code_id').isNull())
        )
        .withColumn('rate_code_id', F.col('rate_code_id').cast('string'))
        .replace(rate_code_id_map, subset='rate_code_id')
        .withColumnRenamed('rate_code_id', 'rate_code')
    )
    # rate_code_id
    df = (
        df.withColumn('rate_code_id', F.col('rate_code'))
        .replace(inverse_rate_code_id_map, subset='rate_code_id')
    )

    # store_and_fwd_flag
    df = (
        df
        .withColumn('store_and_fwd_flag', F.lower(F.trim(F.col('store_and_fwd_flag'))))
        .replace(store_fwd_flag_map, subset='store_and_fwd_flag')
    )
    # store_and_fwd_flag_id
    df = (
        df.withColumn('store_and_fwd_id', F.col('store_and_fwd_flag'))
        .replace(inverse_store_fwd_flag_map, subset='store_and_fwd_id')
    )

    # assign int id for vendor (instead of using string)
    df = (
        df
        .withColumnRenamed('vendor_id', 'vendor')
        .withColumn('vendor', F.lower(F.trim(F.col('vendor'))))
        .replace(vendor_id_map, subset='vendor')
    )
    # vendor_id
    df = (
        df.withColumn('vendor_id', F.col('vendor'))
        .replace(inverse_vendor_id_map, subset='vendor_id')
    )

    # add audit fields of silver zone
    # batch_timestamp = datetime.now(timezone.utc)
    df = (
        df.withColumn('silver_created_datetime', F.current_timestamp())
        .withColumn('silver_updated_datetime', F.current_timestamp())
    )

    # create primary key (sorted columns for deterministic hash)
    # đáng nhé nên thêm `silver_created_datetime` vào `audit_cols` nhưng tôi muốn tạo ra các records bị update giả
    # nếu thêm `silver_created_datetime` thì data sẽ không được update nếu chạy lại (vì đây là dữ liệu tĩnh nên tôi muốn chạy lại để thử logic upsert)
    audit_cols = {'year', 'month', 'source_file', 'source_created_datetime', 'bronze_created_datetime', 'silver_updated_datetime'}
    df = df.withColumn(
        'content_hash',
        F.sha2(F.concat_ws(
            "||",
            *sorted(set(df.columns).difference(audit_cols))
        ), 256)
    )

    # de-dup
    return df.dropDuplicates(['content_hash'])

df_ = _transform(df)

In [10]:
df_.select('vendor', 'vendor_id', 'payment_type', 'payment_type_id', 'rate_code', 'rate_code_id', 'store_and_fwd_flag', 'store_and_fwd_id').dtypes

[('vendor', 'string'),
 ('vendor_id', 'string'),
 ('payment_type', 'string'),
 ('payment_type_id', 'string'),
 ('rate_code', 'string'),
 ('rate_code_id', 'string'),
 ('store_and_fwd_flag', 'string'),
 ('store_and_fwd_id', 'string')]

In [6]:
df_.select('vendor', 'vendor_id', 'payment_type', 'payment_type_id', 'rate_code', 'rate_code_id', 'store_and_fwd_flag', 'store_and_fwd_id').show()

26/10/07 16:14:20 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------------+---------+------------+---------------+-------------+------------+------------------+----------------+
|              vendor|vendor_id|payment_type|payment_type_id|    rate_code|rate_code_id|store_and_fwd_flag|store_and_fwd_id|
+--------------------+---------+------------+---------------+-------------+------------+------------------+----------------+
|  Curb Mobility, LLC|        2| Credit card|              1|Standard rate|           1|                No|               0|
|  Curb Mobility, LLC|        2| Credit card|              1|Standard rate|           1|                No|               0|
|Creative Mobile T...|        1|        Cash|              2|Standard rate|           1|                No|               0|
|  Curb Mobility, LLC|        2| Credit card|              1|Standard rate|           1|                No|               0|
|  Curb Mobility, LLC|        2| Credit card|              1|Standard rate|           1|                No|               0|


In [8]:
value_counts(df, 'payment_type').show()

+------------+------+
|payment_type| count|
+------------+------+
|           4|   527|
|           3|   609|
|           0|  5440|
|           2| 27949|
|           1|105145|
+------------+------+



In [7]:
value_counts(df_, 'payment_type').show()

+------------+------+
|payment_type| count|
+------------+------+
|     Dispute|   512|
|   No charge|   533|
|        Cash| 27192|
| Credit card|102465|
+------------+------+



- Transfer `payment_type`

In [12]:
# replace
df = (
    df
    .withColumn('payment_type', F.lower(F.trim(F.col('payment_type'))))
    .replace(payment_map, subset='payment_type')
)
value_counts(df, 'payment_type').show()

+--------------+------+
|  payment_type| count|
+--------------+------+
|       Dispute|   527|
|     No charge|   609|
|Flex Fare trip|  5440|
|          Cash| 27949|
|   Credit card|105145|
+--------------+------+



In [15]:
df_ = (
    df.withColumn('payment_type_id', F.col('payment_type'))
    .replace(inverse_payment_map, subset='payment_type_id')
)
df_.select('payment_type_id', 'payment_type').show()

+---------------+------------+
|payment_type_id|payment_type|
+---------------+------------+
|              1| Credit card|
|              1| Credit card|
|              1| Credit card|
|              1| Credit card|
|              1| Credit card|
|              1| Credit card|
|              1| Credit card|
|              2|        Cash|
|              1| Credit card|
|              1| Credit card|
|              2|        Cash|
|              1| Credit card|
|              2|        Cash|
|              1| Credit card|
|              4|     Dispute|
|              1| Credit card|
|              1| Credit card|
|              1| Credit card|
|              1| Credit card|
|              1| Credit card|
+---------------+------------+
only showing top 20 rows


- Transfer `store_and_fwd_flag`

In [7]:
store_fwd_flag_map = {
    # yes
    '1': 'Yes',
    '1.0': 'Yes',
    'y': 'Yes',

    # no
    '0': 'No',
    '0.0': 'No',
    'n': 'No',

    # null
    'nan': None,
    '': None,
}

df = (
    df
    .withColumn('store_and_fwd_flag', F.lower(F.trim(F.col('store_and_fwd_flag'))))
    .replace(store_fwd_flag_map, subset='store_and_fwd_flag')
)
value_counts(df, 'store_and_fwd_flag').show()

+------------------+-------+
|store_and_fwd_flag|  count|
+------------------+-------+
|               Yes|  61405|
|                No|3311830|
|              NULL|3685997|
+------------------+-------+



In [8]:
tmp_df = value_counts(df, 'store_and_fwd_flag').toPandas()
len(tmp_df.iloc[:,0])

3

In [9]:
'Yes' in tmp_df.iloc[:,0].tolist()

True

- Transfer `vendor_id`

In [10]:
vendor_id_map = {
    'dds': 'DDS',

    "1": "Creative Mobile Technologies, LLC",
    'cmt': 'Creative Mobile Technologies, LLC',

    'vts': 'Curb Mobility, LLC',
    "2": "Curb Mobility, LLC",

    "6": "Myle Technologies Inc",

    "7": "Helix",
}

df = (
    df
    .withColumn('vendor_id', F.lower(F.trim(F.col('vendor_id'))))
    .replace(vendor_id_map, subset='vendor_id')
)
value_counts(df, 'vendor_id').show()

+--------------------+-------+
|           vendor_id|  count|
+--------------------+-------+
|                 DDS| 132159|
|  Curb Mobility, LLC|3457549|
|Creative Mobile T...|3469524|
+--------------------+-------+



- Viết thêm logic để xử lý trước khi lưu vào Silver zone

## Load to LakeHouse

- Code ở đây đang chỉ insert vào chứ không upsert
- Đọc code upsert trong file `.py`

In [3]:
# spark.sql('drop schema taxi_data_silver cascade')
spark.sql('create database if not exists silver')

DataFrame[]

In [5]:
from delta.tables import DeltaTable

SCHEMA = "silver"
TABLE  = "nyc_taxi"
table_full_name = f'{SCHEMA}.{TABLE}'
# spark._jsparkSession.catalog().tableExists(table_full_name)

# S3_SILVER_PATH = "s3://nblong-data-project/nyc_taxi/silver/"
delta_table = DeltaTable.forName(spark, table_full_name)

In [10]:
df_ = df.select('*').limit(1_000_000)
# df_.show()

In [ ]:
S3_BRONZE_PATH = "s3://nblong-data-project/nyc_taxi/silver/"
SCHEMA = "silver"
TABLE  = "nyc_taxi"

(
    df_
    .write
    .format('delta')
    .partitionBy("year", "month")
    .mode("append")
    .option("path", S3_BRONZE_PATH)
    .saveAsTable(f"{SCHEMA}.{TABLE}")   # đăng ký luôn với Hive metastore
)

print(f"Done. Table registered as '{SCHEMA}.{TABLE}' → {S3_BRONZE_PATH}")

26/06/18 04:34:18 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/06/18 04:34:47 WARN HiveExternalCatalog: Couldn't find corresponding Hive SerDe for data source provider delta. Persisting data source table `spark_catalog`.`taxi_data_silver`.`nyc_taxi` into Hive metastore in Spark SQL specific format, which is NOT compatible with Hive.
26/06/18 04:34:47 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.


Done. Table registered as 'taxi_data_silver.nyc_taxi' → s3://nblong-data-project/nyc_taxi/silver/


---

In [2]:
spark = (
    SparkSession.builder
    .appName('ingest data to silver zone')
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "8g")
    .config("spark.sql.session.timeZone", "UTC")
    # delta lake support
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    # read postgres
    .config(
        'spark.jars',
        """
        ../../../../spark/jar/delta-spark_4.1_2.13-4.3.0.jar,
        ../../../../spark/jar/delta-kernel-api-4.3.0.jar,
        ../../../../spark/jar/delta-kernel-defaults-4.3.0.jar,
        ../../../../spark/jar/delta-storage-4.3.0.jar,
        """
    )
    .getOrCreate()
)
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/09 20:43:09 WARN Utils: Your hostname, debian-nblong, resolves to a loopback address: 127.0.1.1; using 192.168.1.37 instead (on interface wlp0s20f3)
26/08/09 20:43:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/08/09 20:43:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/09 20:43:10 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
from datetime import datetime, timezone

from delta.tables import DeltaTable


In [4]:
df = (
    spark.read.parquet("../../../../data/test/parquet/yellow_tripdata_2010-01.parquet")
    .sample(fraction=0.1)
)
(
    df
    .withColumn('year', F.year('pickup_datetime'))
    .withColumn('month', F.month('pickup_datetime'))
    .withColumn(
        'trip_key',
        F.sha2(F.concat_ws(
            "||",
            "vendor_id", "pickup_datetime", "dropoff_datetime", "passenger_count", "trip_distance", "pickup_longitude", "pickup_latitude", "rate_code", "store_and_fwd_flag", "dropoff_longitude", "dropoff_latitude", "payment_type", "fare_amount", "surcharge", "mta_tax", "tip_amount", "tolls_amount", "total_amount"
        ), 256)
    )
    .write
    .format('delta')
    # .option("delta.enableDeletionVectors", "true")
    .partitionBy("year", "month")
    .mode("append")
    .save('../../../../data/test/parquet/delta')
)

In [9]:
df.createOrReplaceTempView('nyc_taxi')
bla = spark.sql('select pickup_datetime from nyc_taxi limit 1;').collect()
str(bla)

"[Row(pickup_datetime='2010-01-05 19:27:06')]"

In [10]:
bla[0][0]

'2010-01-05 19:27:06'

In [8]:
partitions = df.select("year", "month").distinct().collect()
partition_filter = " OR ".join([
    f"(target.year = {r['year']} AND target.month = {r['month']})"
    for r in partitions
])
partition_filter

'(target.year = 2010 AND target.month = 1)'

In [5]:
target_table = DeltaTable.forPath(spark, "../../../../data/test/parquet/delta")
df = spark.read.format("delta").load("../../../../data/test/parquet/delta")

In [6]:
updated_df = df.dropDuplicates(['trip_key'])
updated_df = updated_df.filter(F.col('pickup_datetime') < '2010-01-02').withColumn(
    'pickup_datetime',
    F.lit(datetime.now(timezone.utc)).cast('timestamp')
)

In [7]:
(
    target_table.alias('target')
    .merge(
        updated_df.alias('source'),
        'target.trip_key = source.trip_key'
    )
    .whenMatchedDelete()
    .execute()
)

26/07/24 07:35:10 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/07/24 07:35:37 WARN MapPartitionsRDD: RDD 39 was locally checkpointed, its lineage has been truncated and cannot be recomputed after unpersisting


DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [35]:
df.count()

1079810

In [ ]:
updated_df.select('*').groupBy('trip_key').count().filter(F.col('count')>1).show()

+--------------------+-----+
|            trip_key|count|
+--------------------+-----+
|468ae251e621b9a08...|    2|
|50760ef38392820c5...|    2|
|d3ce88d98ecfb0792...|    2|
|7ef1b54abe6337780...|    2|
|c0ba38a4b36d596d7...|    2|
|f0f2610b0a2a76ab5...|    2|
|424dc3da0cf6caa9c...|    2|
|b414d51f0a3fd60b8...|    2|
|4e1b4b1f0c4210330...|    2|
|9714fc3f21634e09b...|    2|
|eecc9c1c6e94bab75...|    2|
|1cc927b7f4c674604...|    2|
|fec0dcb5c7d3b767d...|    2|
|7a6e3cd96ed0ef04e...|    2|
|4e3d9c801ec938167...|    2|
|e19b1666217d622ce...|    2|
+--------------------+-----+



In [ ]:
bla = updated_df.groupBy('trip_key').count()
bla.show(4)

+--------------------+-----+
|            trip_key|count|
+--------------------+-----+
|3b192f64956d26445...|    1|
|3d0a8ecd5406d38b5...|    1|
|19bb7d3630ebf2770...|    1|
|fd055638fe2e9b35b...|    1|
+--------------------+-----+
only showing top 4 rows


In [ ]:
bla.filter(F.col('count')>1).count()

16

In [4]:
from datetime import datetime, timezone

df = spark.read.table('bronze.nyc_taxi')
df.createOrReplaceTempView('nyc_taxi')
batch_start_ts = datetime.now(timezone.utc)
df = spark.sql(f'''
    select * from nyc_taxi
    where bronze_created_datetime <= (
        select max(bronze_created_datetime)
        from (
            select * from nyc_taxi
            where bronze_created_datetime > timestamp '2000-01-01 00:00:00.0'
            order by bronze_created_datetime asc
            limit {140_000*3}
        )
    ) and bronze_created_datetime > timestamp '2000-01-01 00:00:00.0'
    and bronze_created_datetime < '{batch_start_ts!s}'
''')

In [5]:
df.count()

558385